# Foundations Summary

Synthesize verified findings from the foundations labs without introducing unmeasured claims.

## Objectives

Connect CPU, memory, CUDA, profiling, and RDMA observations and identify the next experiments they justify.

## Background

System performance emerges from interactions among compute, memory, storage, scheduling, accelerator execution, and communication.

## Prediction

The completed foundations labs will not support a single universal performance bottleneck. Instead, the dominant constraint should depend on workload scale and execution phase:

- small CPU workloads will be sensitive to core selection, affinity, interpreter overhead, and cache locality;
- sufficiently large CPU and GPU workloads will increasingly expose memory bandwidth and access-pattern effects;
- small CUDA operations will be disproportionately affected by launch and synchronization overhead;
- independent GPU work may overlap only when resource use and dependencies leave concurrency available;
- profiling should show that host-side, device-side, and synchronization time must be measured separately;
- communication between two DGX Spark systems will remain much slower than local memory movement, making communication volume and synchronization frequency important for distributed LLM workloads.

Before treating any of these expectations as findings, this notebook will verify which preceding notebooks contain executed measurements and completed interpretations on the current `main` branch.

## Environment

In [ ]:
import os
import platform
import socket
import sys
from pathlib import Path

repository_root = next(
    (
        path
        for path in (Path.cwd(), *Path.cwd().parents)
        if (path / "pyproject.toml").is_file()
    ),
    None,
)
if repository_root is None:
    raise RuntimeError("Run this notebook from within the repository")
if str(repository_root) not in sys.path:
    sys.path.insert(0, str(repository_root))

print(f"Python: {sys.version}")
print(f"Platform: {platform.platform()}")
print(f"Hostname: {socket.gethostname()}")
print(f"Working directory: {os.getcwd()}")

## Experiment

In [ ]:
import json
from pathlib import Path

import pandas as pd


foundations_directory = repository_root / "experiments" / "00-foundations"

source_notebooks = sorted(
    path
    for path in foundations_directory.glob("[0-9][0-9]-*.ipynb")
    if path.name != "10-summary.ipynb"
)

expected_prefixes = [f"{index:02d}" for index in range(10)]
actual_prefixes = [path.name.split("-", maxsplit=1)[0] for path in source_notebooks]

if actual_prefixes != expected_prefixes:
    raise RuntimeError(
        "Expected foundations notebooks 00 through 09, "
        f"but found prefixes {actual_prefixes}"
    )


def cell_source(cell: dict[str, object]) -> str:
    return "".join(cell.get("source", []))


def output_contains_error(cell: dict[str, object]) -> bool:
    return any(
        output.get("output_type") == "error" for output in cell.get("outputs", [])
    )


def section_body(
    cells: list[dict[str, object]],
    heading: str,
) -> str | None:
    lines: list[str] = []
    collecting = False

    for cell in cells:
        if cell.get("cell_type") != "markdown":
            continue

        text = cell_source(cell).strip()
        if not text:
            continue

        cell_lines = text.splitlines()
        first_line = cell_lines[0].strip()

        if collecting and first_line.startswith("## "):
            break

        if first_line == heading:
            collecting = True
            lines.extend(cell_lines[1:])
        elif collecting:
            lines.extend(cell_lines)

    body = "\n".join(lines).strip()
    return body or None


section_names = [
    "Prediction",
    "Observations",
    "Explanation",
    "Connection to LLMs",
    "Further Exploration",
]

notebook_records: list[dict[str, object]] = []
section_records: list[dict[str, object]] = []
loaded_notebooks: dict[str, dict[str, object]] = {}

for notebook_path in source_notebooks:
    notebook = json.loads(notebook_path.read_text())
    loaded_notebooks[notebook_path.name] = notebook

    cells = notebook["cells"]
    code_cells = [cell for cell in cells if cell.get("cell_type") == "code"]
    markdown_cells = [cell for cell in cells if cell.get("cell_type") == "markdown"]

    executed_code_cells = [
        cell
        for cell in code_cells
        if cell.get("execution_count") is not None or bool(cell.get("outputs"))
    ]

    error_cells = [cell for cell in code_cells if output_contains_error(cell)]

    notebook_source = "\n".join(cell_source(cell) for cell in cells)

    extracted_sections = {
        section_name: section_body(
            cells,
            f"## {section_name}",
        )
        for section_name in section_names
    }

    observations = extracted_sections["Observations"]
    explanation = extracted_sections["Explanation"]

    observations_complete = observations is not None and "TODO" not in observations
    explanation_complete = explanation is not None and "TODO" not in explanation

    notebook_records.append(
        {
            "notebook": notebook_path.name,
            "markdown_cells": len(markdown_cells),
            "code_cells": len(code_cells),
            "executed_code_cells": len(executed_code_cells),
            "execution_coverage": (
                len(executed_code_cells) / len(code_cells) if code_cells else None
            ),
            "saved_error_cells": len(error_cells),
            "todo_occurrences": notebook_source.count("TODO"),
            "observations_complete": observations_complete,
            "explanation_complete": explanation_complete,
            "summary_candidate": (observations_complete and explanation_complete),
        }
    )

    for section_name, body in extracted_sections.items():
        section_records.append(
            {
                "notebook": notebook_path.name,
                "section": section_name,
                "present": body is not None,
                "complete": (body is not None and "TODO" not in body),
                "character_count": (len(body) if body is not None else 0),
                "body": body,
            }
        )


notebook_inventory = pd.DataFrame(notebook_records)
recorded_sections = pd.DataFrame(section_records)

notebook_inventory

,notebook,markdown_cells,code_cells,executed_code_cells,execution_coverage,saved_error_cells,todo_occurrences,observations_complete,explanation_complete,summary_candidate
0,00-machine-overview.ipynb,10,2,2,1.000000,0,1,True,True,True
1,01-cpu-architecture.ipynb,10,2,2,1.000000,0,2,True,True,True
2,02-memory-hierarchy.ipynb,10,28,28,1.000000,0,1,True,True,True
3,03-linux-memory.ipynb,10,28,27,0.964286,0,1,True,True,True
4,04-vectorization.ipynb,10,17,17,1.000000,0,1,True,True,True
5,05-cuda-fundamentals.ipynb,18,22,22,1.000000,0,0,True,True,True
6,06-gpu-memory.ipynb,15,18,15,0.833333,0,1,True,True,True
7,07-cuda-streams.ipynb,14,19,17,0.894737,0,0,True,True,True
8,08-profiling.ipynb,27,40,39,0.975000,0,0,True,True,True
9,09-rdma-fundamentals.ipynb,25,69,63,0.913043,0,0,True,True,True


In [ ]:
inventory_summary = pd.Series(
    {
        "source_notebooks": len(notebook_inventory),
        "notebooks_with_saved_outputs": int(
            (notebook_inventory["executed_code_cells"] > 0).sum()
        ),
        "notebooks_with_saved_errors": int(
            (notebook_inventory["saved_error_cells"] > 0).sum()
        ),
        "notebooks_containing_todos": int(
            (notebook_inventory["todo_occurrences"] > 0).sum()
        ),
        "completed_observation_sections": int(
            notebook_inventory["observations_complete"].sum()
        ),
        "completed_explanation_sections": int(
            notebook_inventory["explanation_complete"].sum()
        ),
        "summary_candidates": int(notebook_inventory["summary_candidate"].sum()),
    },
    name="count",
)

inventory_summary.to_frame()

,count
source_notebooks,10
notebooks_with_saved_outputs,10
notebooks_with_saved_errors,0
notebooks_containing_todos,6
completed_observation_sections,10
completed_explanation_sections,10
summary_candidates,10


In [ ]:
section_status = (
    recorded_sections.pivot(
        index="notebook",
        columns="section",
        values="complete",
    )
    .rename_axis(columns=None)
    .reset_index()
)

for section_name in section_names:
    if section_name in section_status:
        section_status[section_name] = section_status[section_name].map(
            {
                True: "complete",
                False: "TODO or missing",
            }
        )

section_status

,notebook,Connection to LLMs,Explanation,Further Exploration,Observations,Prediction
0,00-machine-overview.ipynb,complete,complete,complete,complete,TODO or missing
1,01-cpu-architecture.ipynb,complete,complete,complete,complete,TODO or missing
2,02-memory-hierarchy.ipynb,complete,complete,TODO or missing,complete,complete
3,03-linux-memory.ipynb,complete,complete,TODO or missing,complete,complete
4,04-vectorization.ipynb,complete,complete,TODO or missing,complete,complete
5,05-cuda-fundamentals.ipynb,complete,complete,complete,complete,complete
6,06-gpu-memory.ipynb,complete,complete,TODO or missing,complete,complete
7,07-cuda-streams.ipynb,complete,complete,complete,complete,complete
8,08-profiling.ipynb,complete,complete,complete,complete,complete
9,09-rdma-fundamentals.ipynb,complete,complete,complete,complete,complete


### Evidence-admission criterion

Notebook completion is not inferred from execution-count coverage alone. A notebook is admitted as a summary candidate when:

- its Observations section contains completed prose;
- its Explanation section contains completed prose;
- neither section is still marked as TODO.

This admits the notebook's narrative as a candidate index. It does not automatically validate every claim in that prose. Numerical statements must still be traceable to saved outputs, while causal explanations must remain distinguishable from measurements.

In [ ]:
candidate_notebooks = notebook_inventory.loc[
    notebook_inventory["summary_candidate"],
    "notebook",
].tolist()

excluded_notebooks = notebook_inventory.loc[
    ~notebook_inventory["summary_candidate"],
    [
        "notebook",
        "observations_complete",
        "explanation_complete",
    ],
].reset_index(drop=True)

print(f"Summary candidates: {len(candidate_notebooks)} / {len(notebook_inventory)}")

for notebook in candidate_notebooks:
    print(f"- included: {notebook}")

for row in excluded_notebooks.itertuples(index=False):
    missing_sections = []

    if not row.observations_complete:
        missing_sections.append("Observations")
    if not row.explanation_complete:
        missing_sections.append("Explanation")

    print(f"- excluded: {row.notebook} ({', '.join(missing_sections)} incomplete)")

Summary candidates: 10 / 10
- included: 00-machine-overview.ipynb
- included: 01-cpu-architecture.ipynb
- included: 02-memory-hierarchy.ipynb
- included: 03-linux-memory.ipynb
- included: 04-vectorization.ipynb
- included: 05-cuda-fundamentals.ipynb
- included: 06-gpu-memory.ipynb
- included: 07-cuda-streams.ipynb
- included: 08-profiling.ipynb
- included: 09-rdma-fundamentals.ipynb


In [ ]:
candidate_narrative = (
    recorded_sections[
        recorded_sections["notebook"].isin(candidate_notebooks)
        & recorded_sections["section"].isin(
            [
                "Observations",
                "Explanation",
                "Connection to LLMs",
                "Further Exploration",
            ]
        )
        & recorded_sections["complete"]
    ]
    .copy()
    .sort_values(
        [
            "notebook",
            "section",
        ]
    )
    .reset_index(drop=True)
)

candidate_narrative[
    [
        "notebook",
        "section",
        "character_count",
        "body",
    ]
]

,notebook,section,character_count,body
0,00-machine-overview.ipynb,Connection to LLMs,676,The shared memory capacity sets an upper bound...
1,00-machine-overview.ipynb,Explanation,608,"The results confirm the expected architecture,..."
2,00-machine-overview.ipynb,Further Exploration,231,Map each logical CPU number to its core model ...
3,00-machine-overview.ipynb,Observations,831,- The machine runs 64-bit `aarch64` Linux with...
4,01-cpu-architecture.ipynb,Connection to LLMs,487,"SVE, BF16, and 8-bit matrix instructions can a..."
5,01-cpu-architecture.ipynb,Explanation,536,The matching identifiers show that logical CPU...
6,01-cpu-architecture.ipynb,Further Exploration,305,Read the `CPU part` value for every logical CP...
7,01-cpu-architecture.ipynb,Observations,475,- The saved output shows complete records for ...
8,02-memory-hierarchy.ipynb,Connection to LLMs,110,Memory hierarchy behavior influences embedding...
9,02-memory-hierarchy.ipynb,Explanation,1143,Cache capacity is not a hard threshold at whic...


In [ ]:
from IPython.display import Markdown, display


for notebook in candidate_notebooks:
    display(Markdown(f"### `{notebook}`"))

    notebook_sections = candidate_narrative[candidate_narrative["notebook"] == notebook]

    for row in notebook_sections.itertuples(index=False):
        display(Markdown(f"#### {row.section}\n\n{row.body}"))

### `00-machine-overview.ipynb`

#### Connection to LLMs

The shared memory capacity sets an upper bound for model weights, the key-value cache, activations, and host processes.

The GB10 GPU performs the matrix operations used for token generation. The CPU cores support tokenization, scheduling, and data preparation.

The heterogeneous CPU cores can produce variable preprocessing performance when the operating system moves work between core types.

The NVMe device affects model load time and weight transfer after a model leaves memory. ConnectX-7 can support communication between nodes.

This overview does not measure bandwidth, latency, or token throughput. Later experiments must measure these values under an LLM workload.

#### Explanation

The results confirm the expected architecture, core count, CPU models, NUMA layout, GPU, network adapter, and NVMe storage.

The two CPU model entries describe the heterogeneous core groups. Each entry accounts for 10 of the 20 physical cores.

The expected 128 GB and reported 121 GiB use different units. Platform reservations can also reduce memory visible to Linux.

`nvidia-smi` reports `Not Supported` for GPU memory use. The output therefore does not measure the GB10 unified memory pool.

The PCIe output confirms ConnectX-7 hardware but does not show link speed, link width, or active network ports.

#### Further Exploration

Map each logical CPU number to its core model and maximum frequency through `/sys/devices/system/cpu`.

Then pin the same preprocessing task to each core group with `taskset`. Compare runtime and confirm the performance difference.

#### Observations

- The machine runs 64-bit `aarch64` Linux with kernel `6.17.0-1026-nvidia`.
- The CPU has 20 online cores and one thread per core.
- Ten Cortex-X925 cores reach 3.9 GHz. Ten Cortex-A725 cores reach 2.808 GHz.
- One NUMA node contains all 20 CPU cores and 124,609 MB of memory.
- `free -h` reports 121 GiB total memory, 114 GiB available memory, and 15 GiB swap.
- `nvidia-smi` identifies one NVIDIA GB10 GPU with driver 580.159.03 and CUDA 13.0 support.
- The idle GPU used 4 W, had 0% utilization, and had a temperature of 47 degrees Celsius.
- A 3.7 TB Samsung NVMe device contains the EFI and root partitions.
- PCIe lists four ConnectX-7 Ethernet functions, one Realtek Ethernet controller, and one MediaTek network controller.
- All seven commands completed successfully. No command timed out or reported a missing executable.

...

### `09-rdma-fundamentals.ipynb`

#### Connection to LLMs

Distributed LLM workloads move large tensors between accelerators and systems. The communication can include these operations:

- Tensor-parallel collectives
- Pipeline-parallel activation transfers
- Expert routing for mixture-of-experts models
- Gradient synchronization
- Optimizer-state or parameter transfers
- Distributed checkpoints
- Remote KV-cache access or migration

The notebook measured two communication regimes that affect these workloads.

### Small transfers and synchronization

The 8-byte test measures the fixed cost of a very small RDMA operation. LLM collectives use large buffers but also use small control transfers.

A difference of several tenths of a microsecond has little effect on one operation. It can matter across many sequential communication steps.

Latency is important for these workloads:

- Fine-grained tensor parallelism
- Small microbatches
- Pipeline bubbles
- Frequent expert-routing transfers
- Inference with short sequences or small batches

The CPU results show that host execution affects low-level communication tests. Use Cortex-X925 cores for runtime threads that launch, poll, schedule, or coordinate communication.

### Large tensor transfers

The bandwidth sweep better represents transfers of activations, parameters, gradients, or KV-cache blocks.

One PCI host path supported approximately 110 Gb/s, or 13.7 GB/s. Both paths together supported approximately 196 Gb/s, or 24.5 GB/s.

Use this simplified transfer-time estimate:

$$
t \approx \frac{\text{tensor size}}{\text{effective bandwidth}}
$$

A 1 GiB transfer takes approximately 78 ms at 13.7 GB/s. It takes approximately 44 ms at 24.5 GB/s.

These estimates exclude software overhead, collective structure, contention, and overlap with computation.

Software that uses only one RDMA interface leaves much of the available network capacity unused. An LLM runtime must use both RDMA paths.

The runtime can use one of these methods:

- Divide traffic between both paths.
- Use a transport that knows the dual-path topology.
- Create communication channels with routes across both paths.

More queue pairs on one interface do not solve this problem.

### Communication and computation overlap

LLM systems usually overlap communication with other work:

- Matrix multiplication
- Attention computation
- Expert execution
- Pipeline work between layers
- Host scheduling

Thus, the measured network rate does not directly define model throughput. It defines the communication work that the runtime must hide or distribute.

The DGX Spark 200 Gb/s rating describes its topology. It does not describe one socket or one RDMA interface.

Applications must use both PCI host paths to approach the full bandwidth.

#### Explanation

This notebook measured the RDMA topology and baseline performance of DGX Spark. The tests used host-memory RoCE traffic between two systems.

The local system exposes four RDMA devices. Two devices are active and use Ethernet with RoCE v2:

- `roceP2p1s0f1` uses the `10.200.0.0/30` network.
- `rocep1s0f1` uses the `10.201.0.0/30` network.

The two active interfaces use separate PCI domains. They do not represent two independent 200 Gb/s NICs.

DGX Spark exposes one ConnectX-7 through two PCI host paths. Together, these paths provide the advertised 200 Gb/s NIC capacity.

An RC ping-pong test first confirmed RDMA communication through the configured RoCE routes.

The latency tests used 8-byte RDMA writes. The two PCI paths gave different results:

- The `10.200` path averaged approximately 1.8 µs.
- The `10.201` path averaged approximately 1.4 µs.

Repeated paired trials showed a consistent difference. CPU placement also changed the measured latency.

Cortex-X925 cores gave lower latency than Cortex-A725 cores. A change of CPU cluster without a change of core type had little effect.

Tests with different client and server CPUs showed effects from the benchmark and path. The initiating CPU had more effect on `10.200`.

On `10.201`, a change to either endpoint CPU affected the result. This does not mean that the remote CPU processes each RDMA write.

The complete `ib_write_lat` test includes synchronization and completion work. This work makes the result sensitive to CPU placement at each endpoint.

The message-size sweep showed a change from message-rate limits to bandwidth limits:

- Small messages gave low byte throughput.
- Throughput increased quickly through the kilobyte range.
- Messages from 16 KiB through 64 KiB approached the single-path plateau.
- Large messages reached approximately 109–112 Gb/s on each path.

More queue pairs did not materially increase this limit. Separate processes on different Cortex-X925 cores also did not increase one-path throughput.

Thus, one queue pair, one posting loop, and one CPU core did not cause the one-path limit.

Concurrent use of both PCI paths gave these results:

- `10.200` delivered approximately 98 Gb/s.
- `10.201` delivered approximately 98 Gb/s.
- Total throughput reached approximately 196 Gb/s.

The total is approximately 98% of the advertised 200 Gb/s ConnectX-7 capacity.

The approximately 110 Gb/s one-path plateau is a PCI host-path limit. Applications must use both paths to reach almost all NIC bandwidth.

The tests support these conclusions:

1. DGX Spark provides RoCE v2 through two active RDMA interfaces.
2. The two PCI host paths have different small-message latency.
3. Cortex-X925 placement gives lower latency than Cortex-A725 placement.
4. One PCI path supports approximately 110 Gb/s of large-message RDMA writes.
5. More queue pairs or posting processes do not remove the one-path limit.
6. Concurrent use of both paths gives approximately 196 Gb/s in total.

#### Further Exploration

This notebook measured RDMA latency and bandwidth with host memory. The following tests can answer the remaining questions.

### Reverse-direction measurements

All controlled tests used `spark-0240` as the initiator and `spark-f868` as the target.

Reverse the roles to test whether the latency difference and bandwidth limits are the same in both directions.

### Bidirectional traffic

Use `ib_write_bw --bidirectional` or equivalent paired tests to measure simultaneous traffic in both directions.

These tests can show shared PCI, memory, or NIC resources. They can also test whether each direction provides 200 Gb/s.

### Automatic dual-path traffic

The dual-path experiment started one process for each interface. Test whether common communication libraries can use both paths automatically.

Candidates include these libraries:

- UCX
- NCCL
- MPI implementations with UCX support
- libfabric
- Distributed PyTorch communication backends

Test whether one logical collective can approach 196 Gb/s without application-managed traffic division.

### RDMA operation types

Compare these additional operations:

- RDMA read latency and bandwidth
- Send and receive operations
- Atomic operations
- Different completion and queue-depth settings

These operations use different NIC and endpoint functions. They can show other CPU or PCI-path effects.

### Queue depth and completion moderation

More queue pairs did not increase single-path bandwidth. Other `perftest` settings can still affect performance:

- Transmit depth
- Number of outstanding requests
- Inline-data thresholds
- Completion-queue moderation
- Event completions compared with polling

These tests can identify the cause of the small increase from one to two queue pairs.

### CPU use and power

The benchmarks measured transfer performance but not its CPU cost.

Measure core use, cycles, instructions, and power. Compare RDMA offload with the cost of benchmark polling.

### GPUDirect RDMA

The current tests used host memory. Distributed accelerator workloads need direct transfers between GPU-accessible memory and the NIC.

Answer these questions:

- Does DGX Spark support and enable GPUDirect RDMA?
- Can the system register unified-memory buffers directly?
- Do transfers avoid intermediate CPU copies?
- How do GPU-memory latency and bandwidth compare with host-memory results?
- Can GPU traffic use both PCI host paths at the same time?

### End-to-end collective performance

Microbenchmarks isolate the transport. LLM systems use all-reduce, all-gather, reduce-scatter, and all-to-all collectives.

Measure these operations across two DGX Spark systems:

1. Run an NCCL point-to-point transfer.
2. Run an NCCL all-reduce operation.
3. Run a tensor-parallel matrix multiplication.
4. Run a pipeline-parallel activation transfer.
5. Run a mixture-of-experts all-to-all operation.

These tests show how much of the 196 Gb/s dual-path result an LLM runtime can use.

#### Observations

Run the cells on one DGX Spark. Keep all output before you make conclusions.

Answer these questions first:

1. Which RDMA tools are installed?
2. Which RDMA devices are visible?
3. Which kernel driver and PCI device support each RDMA device?
4. Which Linux network interfaces map to the RDMA devices?
5. Which link layer does each RDMA port use?
6. Which ports are active, and which nominal rate does each port report?
7. Are the related network interfaces active and assigned an IP address?
8. Does the routing table contain routes through these interfaces?

This inventory alone does not support conclusions about bandwidth, latency, CPU use, or GPUDirect RDMA.

### Classification used in the synthesis

Candidate statements will be classified as:

1. **Measured** — directly visible in a saved output from the originating notebook.
2. **Derived** — calculated from saved measurements using an explicit transformation.
3. **Architectural inference** — an explanation based on observed behavior and system structure, but not directly measured.
4. **Unresolved** — suggested by the experiment but not established by the saved outputs.

Execution coverage and TODO counts are retained as provenance metadata. They are not themselves performance findings.

### Evidence ledger

The narrative sections identify candidate conclusions, but the final synthesis requires a smaller set of traceable claims.

Each ledger entry records:

- the originating notebook;
- the subsystem or performance layer;
- the claim being considered;
- whether it is measured, derived, inferred, or unresolved;
- the saved output or transformation that supports it;
- its relevance to later LLM experiments.

The ledger is intentionally selective. It captures the findings with the greatest cross-layer explanatory value rather than reproducing every benchmark result.

In [16]:
def validate_evidence_ledger(
    ledger: pd.DataFrame,
    candidate_notebooks: list[str],
) -> None:
    missing_columns = [
        column for column in evidence_columns if column not in ledger.columns
    ]
    if missing_columns:
        raise ValueError(f"Missing evidence columns: {missing_columns}")

    unknown_notebooks = sorted(
        set(ledger["notebook"].dropna()) - set(candidate_notebooks)
    )
    if unknown_notebooks:
        raise ValueError(
            f"Evidence refers to non-candidate notebooks: {unknown_notebooks}"
        )

    unknown_classifications = sorted(
        set(ledger["classification"].dropna()) - allowed_classifications
    )
    if unknown_classifications:
        raise ValueError(f"Unknown evidence classifications: {unknown_classifications}")

    duplicated_claims = ledger.duplicated(
        subset=["notebook", "claim"],
        keep=False,
    )
    if duplicated_claims.any():
        duplicates = ledger.loc[
            duplicated_claims,
            ["notebook", "claim"],
        ]
        raise ValueError(
            f"Duplicate evidence claims:\n{duplicates.to_string(index=False)}"
        )

In [17]:
evidence_columns = [
    "notebook",
    "layer",
    "claim",
    "classification",
    "support",
    "llm_relevance",
]

allowed_classifications = {
    "measured",
    "derived",
    "architectural inference",
    "unresolved",
}

evidence_ledger = pd.DataFrame(
    [
        {
            "notebook": "00-machine-overview.ipynb",
            "layer": "machine topology",
            "claim": (
                "The recorded system has 20 online ARM CPU cores, "
                "split into ten Cortex-X925 and ten Cortex-A725 cores, "
                "with one hardware thread per core."
            ),
            "classification": "measured",
            "support": (
                "`lscpu` reports 20 online CPUs, one thread per core, "
                "ten Cortex-X925 cores, and ten Cortex-A725 cores."
            ),
            "llm_relevance": (
                "CPU-side tokenization, request handling, sampling, and "
                "data preparation may run on two materially different "
                "core classes."
            ),
        },
        {
            "notebook": "00-machine-overview.ipynb",
            "layer": "memory topology",
            "claim": (
                "Linux exposes one NUMA node containing all 20 CPU cores "
                "and approximately 121 GiB of total memory."
            ),
            "classification": "measured",
            "support": (
                "`numactl --hardware` reports one NUMA node with CPUs "
                "0-19 and 124,609 MB; `free -h` reports 121 GiB total."
            ),
            "llm_relevance": (
                "Model weights, KV cache, CPU allocations, and GPU-visible "
                "unified-memory pressure ultimately share one system-level "
                "memory capacity."
            ),
        },
        {
            "notebook": "00-machine-overview.ipynb",
            "layer": "accelerator",
            "claim": (
                "The recorded accelerator is one NVIDIA GB10 using driver "
                "580.159.03 with CUDA 13.0 support."
            ),
            "classification": "measured",
            "support": (
                "`nvidia-smi` identifies NVIDIA GB10, driver 580.159.03, "
                "and CUDA version 13.0."
            ),
            "llm_relevance": (
                "This establishes the accelerator and software baseline "
                "under which the later CUDA and GPU-memory measurements "
                "were collected."
            ),
        },
        {
            "notebook": "00-machine-overview.ipynb",
            "layer": "unified memory reporting",
            "claim": (
                "`nvidia-smi` does not expose a conventional dedicated-GPU "
                "memory-usage counter on this GB10 system."
            ),
            "classification": "measured",
            "support": (
                "The saved `nvidia-smi` output reports `Not Supported` "
                "for memory usage."
            ),
            "llm_relevance": (
                "GPU memory capacity and pressure cannot be inferred from "
                "the usual discrete-GPU `nvidia-smi` memory field and need "
                "process- or framework-level measurement."
            ),
        },
        {
            "notebook": "01-cpu-architecture.ipynb",
            "layer": "CPU instruction support",
            "claim": (
                "The displayed records for logical CPUs 0 and 1 report "
                "SVE, SVE2, BF16, and 8-bit matrix instruction support."
            ),
            "classification": "measured",
            "support": (
                "The saved `/proc/cpuinfo` excerpt contains matching "
                "feature lists for complete records of CPUs 0 and 1."
            ),
            "llm_relevance": (
                "Compatible CPU kernels may use these instruction classes "
                "for preprocessing or inference, although this notebook "
                "does not measure their performance."
            ),
        },
        {
            "notebook": "01-cpu-architecture.ipynb",
            "layer": "CPU heterogeneity",
            "claim": (
                "The current CPU-architecture experiment does not establish "
                "a performance difference between core classes."
            ),
            "classification": "unresolved",
            "support": (
                "The run prints only a truncated `/proc/cpuinfo` excerpt; "
                "it sets no affinity and records no workload timings."
            ),
            "llm_relevance": (
                "Any claim about placing latency-sensitive LLM host work "
                "on Cortex-X925 rather than Cortex-A725 cores requires a "
                "separate affinity-controlled benchmark."
            ),
        },
    ],
    columns=evidence_columns,
)

In [18]:
validate_evidence_ledger(
    evidence_ledger,
    candidate_notebooks,
)

print(f"Validated {len(evidence_ledger)} evidence entries.")

evidence_ledger

Validated 6 evidence entries.


,notebook,layer,claim,classification,support,llm_relevance
0,00-machine-overview.ipynb,machine topology,The recorded system has 20 online ARM CPU core...,measured,"`lscpu` reports 20 online CPUs, one thread per...","CPU-side tokenization, request handling, sampl..."
1,00-machine-overview.ipynb,memory topology,Linux exposes one NUMA node containing all 20 ...,measured,`numactl --hardware` reports one NUMA node wit...,"Model weights, KV cache, CPU allocations, and ..."
2,00-machine-overview.ipynb,accelerator,The recorded accelerator is one NVIDIA GB10 us...,measured,"`nvidia-smi` identifies NVIDIA GB10, driver 58...",This establishes the accelerator and software ...
3,00-machine-overview.ipynb,unified memory reporting,`nvidia-smi` does not expose a conventional de...,measured,The saved `nvidia-smi` output reports `Not Sup...,GPU memory capacity and pressure cannot be inf...
4,01-cpu-architecture.ipynb,CPU instruction support,The displayed records for logical CPUs 0 and 1...,measured,The saved `/proc/cpuinfo` excerpt contains mat...,Compatible CPU kernels may use these instructi...
5,01-cpu-architecture.ipynb,CPU heterogeneity,The current CPU-architecture experiment does n...,unresolved,The run prints only a truncated `/proc/cpuinfo...,Any claim about placing latency-sensitive LLM ...


### Initial evidence interpretation

The first two notebooks establish different kinds of evidence.

`00-machine-overview.ipynb` provides a measured platform baseline: heterogeneous CPU core groups, one NUMA domain, Linux-visible memory capacity, and the NVIDIA GB10 software environment.

`01-cpu-architecture.ipynb` is more limited. It confirms instruction features for the displayed logical CPUs, but it does not perform the affinity-controlled timing experiment needed to quantify heterogeneous-core performance. Core-placement effects therefore remain unresolved in the current repository record.

In [20]:
memory_evidence = pd.DataFrame(
    [
        {
            "notebook": "02-memory-hierarchy.ipynb",
            "layer": "CPU cache topology",
            "claim": (
                "The representative CPUs have 64 KiB private L1 data "
                "caches and 2 MiB private L2 caches, but belong to "
                "different shared-L3 domains: 8 MiB for CPUs 0-9 and "
                "16 MiB for CPUs 10-19."
            ),
            "classification": "measured",
            "support": (
                "The sysfs cache inventory reports CPU 5 with an "
                "8 MiB L3 shared by CPUs 0-9 and CPU 15 with a "
                "16 MiB L3 shared by CPUs 10-19."
            ),
            "llm_relevance": (
                "CPU-side working sets such as tokenizer tables, request "
                "metadata, and preprocessing buffers can experience "
                "different cache capacity depending on core placement."
            ),
        },
        {
            "notebook": "02-memory-hierarchy.ipynb",
            "layer": "memory latency",
            "claim": (
                "Dependent random-load latency increased from about "
                "1.3 ns for small cache-resident working sets to about "
                "4 ns in the L2 region, about 6 ns in shared cache, and "
                "tens to roughly 100 ns for large working sets."
            ),
            "classification": "measured",
            "support": (
                "The pointer-chasing benchmark reports distinct latency "
                "regimes as working-set size crosses private cache, "
                "shared cache, and main-memory scales."
            ),
            "llm_relevance": (
                "Latency-sensitive pointer-heavy host work cannot be "
                "reasoned about from arithmetic throughput alone; "
                "working-set size and locality materially affect it."
            ),
        },
        {
            "notebook": "02-memory-hierarchy.ipynb",
            "layer": "CPU memory bandwidth",
            "claim": (
                "A vectorized sequential read reached about 124-125 GB/s "
                "for cache-resident arrays but only about 32-35 GB/s for "
                "a 256 MiB DRAM-resident working set."
            ),
            "classification": "measured",
            "support": (
                "The eight-accumulator loop was vectorized with 128-bit "
                "vectors; recorded bandwidth remained near 124-125 GB/s "
                "for small working sets and fell to 32-35 GB/s at "
                "256 MiB."
            ),
            "llm_relevance": (
                "CPU kernels can be compute- or cache-throughput-bound "
                "for small data but become constrained by sustainable "
                "memory bandwidth once their working set reaches DRAM."
            ),
        },
        {
            "notebook": "02-memory-hierarchy.ipynb",
            "layer": "CPU optimization",
            "claim": (
                "Removing a single reduction dependency and enabling "
                "vectorization increased cache-resident sequential-read "
                "throughput from about 31 GB/s to about 124-125 GB/s."
            ),
            "classification": "derived",
            "support": (
                "The recorded scalar result is approximately 31 GB/s "
                "and the vectorized eight-accumulator result is "
                "approximately 124-125 GB/s, implying roughly a "
                "fourfold increase for cache-resident data."
            ),
            "llm_relevance": (
                "Instruction-level parallelism and vectorization can "
                "matter as much as core frequency for CPU-side kernels "
                "that have sufficient locality."
            ),
        },
        {
            "notebook": "03-linux-memory.ipynb",
            "layer": "virtual memory",
            "claim": (
                "Reserving a 1 GiB anonymous mapping increased process "
                "virtual size by exactly 1024 MiB while increasing RSS "
                "by only 0.19 MiB."
            ),
            "classification": "measured",
            "support": (
                "`VmSize` rose from 2634.42 MiB to 3658.42 MiB, while "
                "`RssAnon` and smaps anonymous memory did not increase."
            ),
            "llm_relevance": (
                "A large virtual allocation or reservation does not by "
                "itself imply that model or KV-cache pages consume the "
                "same amount of physical memory."
            ),
        },
        {
            "notebook": "03-linux-memory.ipynb",
            "layer": "demand paging",
            "claim": (
                "Touching one byte in every 4 KiB page of the reserved "
                "1 GiB mapping caused RSS to increase by 1024.19 MiB and "
                "anonymous RSS by exactly 1024 MiB."
            ),
            "classification": "measured",
            "support": (
                "Each additional 128 MiB of pages touched produced an "
                "approximately 128 MiB increase in RSS and anonymous RSS."
            ),
            "llm_relevance": (
                "Physical memory pressure appears when pages are first "
                "accessed, so allocation timing and first-touch behavior "
                "can affect model-loading latency and observed memory use."
            ),
        },
        {
            "notebook": "03-linux-memory.ipynb",
            "layer": "page tables",
            "claim": (
                "Making the 1 GiB mapping resident increased the "
                "system page-table counter by approximately 2.1 MiB."
            ),
            "classification": "measured",
            "support": (
                "The recorded system PageTables counter increased by "
                "about 2.1 MiB while 262,144 individual 4 KiB pages "
                "were resident."
            ),
            "llm_relevance": (
                "Large numbers of small mappings or pages create memory-"
                "management overhead in addition to the payload data."
            ),
        },
        {
            "notebook": "03-linux-memory.ipynb",
            "layer": "page tables",
            "claim": (
                "The observed approximately 2.1 MiB page-table increase "
                "is consistent with roughly 2 MiB of leaf page-table "
                "entries for 262,144 mapped 4 KiB pages."
            ),
            "classification": "architectural inference",
            "support": (
                "262,144 pages multiplied by an 8-byte page-table entry "
                "equals 2 MiB; the measured counter rose by about "
                "2.1 MiB."
            ),
            "llm_relevance": (
                "The page granularity used for large model mappings can "
                "influence translation structures and memory-management "
                "overhead."
            ),
        },
    ],
    columns=evidence_columns,
)

evidence_ledger = pd.concat(
    [
        evidence_ledger,
        memory_evidence,
    ],
    ignore_index=True,
)

validate_evidence_ledger(
    evidence_ledger,
    candidate_notebooks,
)

evidence_ledger

,notebook,layer,claim,classification,support,llm_relevance
0,00-machine-overview.ipynb,machine topology,The recorded system has 20 online ARM CPU core...,measured,"`lscpu` reports 20 online CPUs, one thread per...","CPU-side tokenization, request handling, sampl..."
1,00-machine-overview.ipynb,memory topology,Linux exposes one NUMA node containing all 20 ...,measured,`numactl --hardware` reports one NUMA node wit...,"Model weights, KV cache, CPU allocations, and ..."
2,00-machine-overview.ipynb,accelerator,The recorded accelerator is one NVIDIA GB10 us...,measured,"`nvidia-smi` identifies NVIDIA GB10, driver 58...",This establishes the accelerator and software ...
3,00-machine-overview.ipynb,unified memory reporting,`nvidia-smi` does not expose a conventional de...,measured,The saved `nvidia-smi` output reports `Not Sup...,GPU memory capacity and pressure cannot be inf...
4,01-cpu-architecture.ipynb,CPU instruction support,The displayed records for logical CPUs 0 and 1...,measured,The saved `/proc/cpuinfo` excerpt contains mat...,Compatible CPU kernels may use these instructi...
5,01-cpu-architecture.ipynb,CPU heterogeneity,The current CPU-architecture experiment does n...,unresolved,The run prints only a truncated `/proc/cpuinfo...,Any claim about placing latency-sensitive LLM ...
6,02-memory-hierarchy.ipynb,CPU cache topology,The representative CPUs have 64 KiB private L1...,measured,The sysfs cache inventory reports CPU 5 with a...,CPU-side working sets such as tokenizer tables...
7,02-memory-hierarchy.ipynb,memory latency,Dependent random-load latency increased from a...,measured,The pointer-chasing benchmark reports distinct...,Latency-sensitive pointer-heavy host work cann...
8,02-memory-hierarchy.ipynb,CPU memory bandwidth,A vectorized sequential read reached about 124...,measured,The eight-accumulator loop was vectorized with...,CPU kernels can be compute- or cache-throughpu...
9,02-memory-hierarchy.ipynb,CPU optimization,Removing a single reduction dependency and ena...,derived,The recorded scalar result is approximately 31...,Instruction-level parallelism and vectorizatio...


### Memory-layer interpretation

The CPU memory results show that “CPU performance” is not one quantity. Dependent access latency, sequential bandwidth, cache capacity, instruction-level parallelism, and vectorization expose different limits.

The Linux memory experiment adds a second distinction: virtual address-space size, resident physical memory, and system-wide available memory are related but not interchangeable. A one-GiB mapping consumed almost no physical memory until its pages were touched.

Together, these results imply that later LLM experiments must report both access behavior and allocation state. A model can have a large virtual footprint without all pages being resident, while a resident workload can still behave very differently depending on whether its active working set is served from cache or DRAM.

In [22]:
compute_evidence = pd.DataFrame(
    [
        {
            "notebook": "04-vectorization.ipynb",
            "layer": "Python overhead",
            "claim": (
                "For a 16-element multiply-add, the Python loop was "
                "faster than NumPy because NumPy's fixed dispatch cost "
                "exceeded the short loop's execution cost."
            ),
            "classification": "measured",
            "support": (
                "The size sweep records Python ahead at 16 elements; "
                "NumPy overtakes it by 64 elements."
            ),
            "llm_relevance": (
                "Moving tiny host operations into an array library or "
                "accelerator kernel is not automatically beneficial when "
                "dispatch overhead exceeds useful work."
            ),
        },
        {
            "notebook": "04-vectorization.ipynb",
            "layer": "interpreter overhead",
            "claim": (
                "For larger arrays, NumPy exceeded the Python loop by "
                "more than 100 times."
            ),
            "classification": "measured",
            "support": (
                "The recorded Python implementation required about "
                "23-40 ns per element, while NumPy reached approximately "
                "0.18-0.34 ns per element for larger inputs."
            ),
            "llm_relevance": (
                "Per-element Python work is unsuitable for throughput-"
                "sensitive preprocessing and tensor manipulation; loops "
                "should be moved into compiled or fused kernels."
            ),
        },
        {
            "notebook": "04-vectorization.ipynb",
            "layer": "SIMD",
            "claim": (
                "The largest measured SIMD speedup over the compiled "
                "scalar kernel was 1.86 times at 262,144 elements."
            ),
            "classification": "derived",
            "support": (
                "The scalar kernel took 70.90 microseconds and the "
                "vectorized kernel took 38.19 microseconds."
            ),
            "llm_relevance": (
                "SIMD can materially accelerate cache-resident CPU kernels "
                "used in tokenization, quantization, sampling, or data "
                "format conversion."
            ),
        },
        {
            "notebook": "04-vectorization.ipynb",
            "layer": "memory bandwidth",
            "claim": (
                "The SIMD advantage declined to only 1.05 times for the "
                "128 MiB working set."
            ),
            "classification": "derived",
            "support": (
                "At the largest tested size, scalar and vectorized kernels "
                "both processed approximately 3.2-3.4 billion elements "
                "per second."
            ),
            "llm_relevance": (
                "Once a CPU kernel is bandwidth-bound, wider vector "
                "execution provides little benefit unless data movement "
                "is reduced."
            ),
        },
        {
            "notebook": "04-vectorization.ipynb",
            "layer": "kernel fusion",
            "claim": (
                "A fused compiled multiply-add can outperform separate "
                "NumPy expressions by completing the operation in one "
                "memory traversal."
            ),
            "classification": "architectural inference",
            "support": (
                "The notebook observes that separate NumPy ufuncs may "
                "require multiple passes and temporary-array management, "
                "while the compiled kernel fuses the operations."
            ),
            "llm_relevance": (
                "Fusion is valuable in LLM runtimes because avoiding "
                "intermediate tensors reduces memory traffic and launch "
                "overhead."
            ),
        },
        {
            "notebook": "05-cuda-fundamentals.ipynb",
            "layer": "CUDA timing",
            "claim": (
                "The Python enqueue duration for a 64 MiB CUDA "
                "multiply-add was much shorter than the synchronized "
                "host duration."
            ),
            "classification": "measured",
            "support": (
                "Across 100 runs, mean enqueue time was approximately "
                "9.86 microseconds while mean synchronized time was "
                "approximately 1247.91 microseconds."
            ),
            "llm_relevance": (
                "Unsynchronized Python timing substantially understates "
                "completed GPU work and cannot be used directly as kernel "
                "or inference latency."
            ),
        },
        {
            "notebook": "05-cuda-fundamentals.ipynb",
            "layer": "CUDA cold start",
            "claim": (
                "The first CUDA multiplication in a fresh subprocess was "
                "approximately 10.3-10.6 times slower than its warmed-up "
                "multiplication."
            ),
            "classification": "measured",
            "support": (
                "The subprocess repetitions consistently recorded a "
                "roughly tenfold first-use slowdown."
            ),
            "llm_relevance": (
                "Model-serving latency should be measured after explicit "
                "runtime, allocator, and kernel warm-up, while cold-start "
                "latency should be reported separately."
            ),
        },
        {
            "notebook": "05-cuda-fundamentals.ipynb",
            "layer": "CUDA Graph submission",
            "claim": (
                "For a fixed sequence of 64 small additions, CUDA Graph "
                "replay reduced host enqueue time from 196.937 "
                "microseconds to approximately 2 microseconds."
            ),
            "classification": "measured",
            "support": (
                "Graph replay enqueue remained about 1.98-2.06 "
                "microseconds, producing a 98.47-times host-submission "
                "speedup at 64 operations."
            ),
            "llm_relevance": (
                "Autoregressive decoding repeatedly submits many short "
                "kernels, so graph replay can reduce CPU launch overhead "
                "when shapes and memory addresses are stable."
            ),
        },
        {
            "notebook": "05-cuda-fundamentals.ipynb",
            "layer": "CUDA Graph execution",
            "claim": (
                "For the 64-operation sequence, CUDA Graph replay reduced "
                "completed CUDA-event time from 247.792 microseconds to "
                "121.440 microseconds."
            ),
            "classification": "measured",
            "support": (
                "The recorded event timings imply approximately a "
                "2.04-times reduction in completed sequence duration."
            ),
            "llm_relevance": (
                "Graph replay can reduce not only Python submission cost "
                "but also inter-kernel gaps that contribute directly to "
                "token latency."
            ),
        },
        {
            "notebook": "05-cuda-fundamentals.ipynb",
            "layer": "CUDA Graph applicability",
            "claim": (
                "CUDA Graph benefits require stable operation topology, "
                "tensor shapes, and compatible memory addresses."
            ),
            "classification": "architectural inference",
            "support": (
                "Graph capture replays a previously recorded fixed "
                "sequence rather than reconstructing arbitrary Python "
                "control flow."
            ),
            "llm_relevance": (
                "Graph capture is most applicable to stable decode buckets "
                "and less applicable when batches, shapes, or control flow "
                "change on every request."
            ),
        },
    ],
    columns=evidence_columns,
)

evidence_ledger = pd.concat(
    [
        evidence_ledger,
        compute_evidence,
    ],
    ignore_index=True,
)

validate_evidence_ledger(
    evidence_ledger,
    candidate_notebooks,
)

evidence_ledger

,notebook,layer,claim,classification,support,llm_relevance
0,00-machine-overview.ipynb,machine topology,The recorded system has 20 online ARM CPU core...,measured,"`lscpu` reports 20 online CPUs, one thread per...","CPU-side tokenization, request handling, sampl..."
1,00-machine-overview.ipynb,memory topology,Linux exposes one NUMA node containing all 20 ...,measured,`numactl --hardware` reports one NUMA node wit...,"Model weights, KV cache, CPU allocations, and ..."
2,00-machine-overview.ipynb,accelerator,The recorded accelerator is one NVIDIA GB10 us...,measured,"`nvidia-smi` identifies NVIDIA GB10, driver 58...",This establishes the accelerator and software ...
3,00-machine-overview.ipynb,unified memory reporting,`nvidia-smi` does not expose a conventional de...,measured,The saved `nvidia-smi` output reports `Not Sup...,GPU memory capacity and pressure cannot be inf...
4,01-cpu-architecture.ipynb,CPU instruction support,The displayed records for logical CPUs 0 and 1...,measured,The saved `/proc/cpuinfo` excerpt contains mat...,Compatible CPU kernels may use these instructi...
5,01-cpu-architecture.ipynb,CPU heterogeneity,The current CPU-architecture experiment does n...,unresolved,The run prints only a truncated `/proc/cpuinfo...,Any claim about placing latency-sensitive LLM ...
6,02-memory-hierarchy.ipynb,CPU cache topology,The representative CPUs have 64 KiB private L1...,measured,The sysfs cache inventory reports CPU 5 with a...,CPU-side working sets such as tokenizer tables...
7,02-memory-hierarchy.ipynb,memory latency,Dependent random-load latency increased from a...,measured,The pointer-chasing benchmark reports distinct...,Latency-sensitive pointer-heavy host work cann...
8,02-memory-hierarchy.ipynb,CPU memory bandwidth,A vectorized sequential read reached about 124...,measured,The eight-accumulator loop was vectorized with...,CPU kernels can be compute- or cache-throughpu...
9,02-memory-hierarchy.ipynb,CPU optimization,Removing a single reduction dependency and ena...,derived,The recorded scalar result is approximately 31...,Instruction-level parallelism and vectorizatio...


### Compute and submission interpretation

The vectorization and CUDA results expose the same scale-dependent pattern at two different layers.

For very small operations, fixed dispatch or launch overhead can exceed the useful computation. As work increases, moving execution into compiled, SIMD, or GPU kernels becomes highly advantageous. At still larger scales, the benefit can flatten when memory movement rather than arithmetic becomes dominant.

CUDA adds an asynchronous submission layer. A short Python call can represent only enqueue time, while the actual GPU work completes later. CUDA Graph replay demonstrates that repeated submission itself can become a bottleneck: bundling a stable sequence reduced host submission cost by almost two orders of magnitude and reduced completed execution time by about twofold for the longest tested sequence.

These results support treating LLM execution as a pipeline of distinct costs: Python or framework dispatch, kernel launch, device execution, synchronization, and memory traffic.

In [23]:
ledger_coverage = (
    evidence_ledger.groupby(
        [
            "notebook",
            "classification",
        ],
        observed=True,
    )
    .size()
    .rename("claims")
    .reset_index()
    .sort_values(
        [
            "notebook",
            "classification",
        ]
    )
    .reset_index(drop=True)
)

covered_notebooks = evidence_ledger["notebook"].nunique()

print(f"Evidence entries: {len(evidence_ledger)}")
print(f"Notebooks covered: {covered_notebooks} / {len(candidate_notebooks)}")

ledger_coverage

Evidence entries: 24
Notebooks covered: 6 / 10


,notebook,classification,claims
0,00-machine-overview.ipynb,measured,4
1,01-cpu-architecture.ipynb,measured,1
2,01-cpu-architecture.ipynb,unresolved,1
3,02-memory-hierarchy.ipynb,derived,1
4,02-memory-hierarchy.ipynb,measured,3
5,03-linux-memory.ipynb,architectural inference,1
6,03-linux-memory.ipynb,measured,3
7,04-vectorization.ipynb,architectural inference,1
8,04-vectorization.ipynb,derived,2
9,04-vectorization.ipynb,measured,2


## Observations

TODO: Summarize measured findings and link each statement to its originating run.

## Explanation

TODO: Build a cross-layer explanation and distinguish evidence from remaining hypotheses.

## Connection to LLMs

Use the verified constraints to motivate concrete experiments in distributed inference, training, and communication.

## Further Exploration

TODO: Rank follow-up experiments by expected learning value, cost, and dependency on unresolved questions.